# 🥁 DrumScript: Audio to Sheet Music & Stem Separation

<!--date_created: sat-17-january-2026-->
<!--date_updated: sun-01-march-2026-->

Welcome to the interactive demo for **DrumScript**!

This notebook demonstrates the core capabilities of the library:
1.  **Transcription:** Converting drum audio into a PDF Score.
2.  **Stem Splitting:** Extracting drums from a full song (see also notebook `extract_stems.ipynb`)
3.  **Backing Tracks:** Creating "drumless" play-along tracks.
4.  **Analysis:** Detecting tempo and onsets using Python.

> **Note:** This notebook runs on a remote server. To hear audio, we will use the notebook's built-in audio player.

In [ ]:
"""# @title 🛠️ Setup & Installation
# 1. Install System Dependencies (FFmpeg for audio, LilyPond for sheet music)
print("Installing system dependencies...")
!apt-get update -qq
!apt-get install -y ffmpeg lilypond libportaudio2

# 2. Install DrumScript (from PyPI)
print("Installing DrumScript...")
!uv pip install drumscript

# 3. Import Libraries
import drumscript as ds
from ds.audio_processor.stem_splitter import separate_audio
import IPython.display as ipd
import os

print("Setup Complete!")"""

In [ ]:
"""# @title Download Sample Audio
# We'll download a sample drum beat to test with.
# (You can also upload your own file to the 'Files' tab on the left!)

sample_url = "https://github.com/DrumScript/DrumScript/raw/main/tests/data/test_drum_loop.wav"
output_file = "demo_beat.wav"

if not os.path.exists(output_file):
    !wget -q {sample_url} -O {output_file}
    print(f"Downloaded {output_file}")
else:
    print(f"{output_file} already exists")

# Listen to the input
print("Input Audio:")
ipd.Audio(output_file)"""


## 1. `DrumScript` Core Feature: Audio to Sheet Music

The primary goal of DrumScript is to generate readable sheet music. We can run the full pipeline using the Command Line Interface (CLI).

In [ ]:
# Run the main script on our demo file
# This will: Load -> Detect Tempo -> Classify Hits -> Generate PDF
!python -m drumscript.main "demo_beat.wav" --ts 4/4

print("\nTranscription Complete. Check the 'outputs' folder in the file browser on the left for your PDF!")


## 2. Creating Backing Tracks (Stem Splitting)

DrumScript can also remove drums from a song so you can play along.


In [ ]:

# Generate Drumless Track
# Use the Python API to separate the audio
print("Separating stems... (This may take a moment for the model to run)")

# This extracts the 'no_drums' stem (Everything minus drums)
stems = separate_audio("demo_beat.wav", drumless=True, output_dir="stems")

print(f"Stems saved to: {stems}")

# Listen to the result
print("🎧 Drumless Track (Backing Track):")
# Note: Since our demo file is just drums, this might be silent!
# Try uploading a full song to hear this properly.
if 'no_drums' in stems:
    display(ipd.Audio(stems['no_drums']))

## 3. Using the Python API

For developers, DrumScript exposes powerful tools for audio analysis. You don't have to run the full pipeline; you can just use the components you need.

In [ ]:
# 1. Load Audio using the DrumScript Loader (Handles mono conversion & normalization)
import drumscript as ds

y, sr = ds.load_audio("demo_beat.wav")

# 2. Detect Tempo
bpm = ds.detect_tempo(y, sr)
print(f"Detected Tempo: {bpm:.1f} BPM")

# 3. Detect Onsets (Hit locations)
# We can access the internal processor directly
from drumscript.audio_processor import onset_detector

onsets = onset_detector.detect_onsets(y, sr)
print(f"Detected {len(onsets)} drum hits")


## 4. ⚙️ Transparency: Constants

DrumScript is not a "Black Box." We expose our configuration constants so you know exactly how the audio is being processed (Sample Rates, FFT window sizes, etc.).

In [ ]:
from drumscript.notation_generator import constants

print(f"Global Sample Rate: {constants.SAMPLE_RATE} Hz")
print(f"FFT Hop Length:     {constants.HOP_LENGTH}")
